# Cloud Provider Analytics — ejecución del pipeline en Google Colab

Ejecutar las celdas en orden en un runtime nuevo (*Entorno de ejecución → Ejecutar todas*). El notebook sólo prepara el entorno y llama al pipeline: la lógica está en `src/`.

1. Clona el repositorio en la rama `BRANCH`.
2. Instala Java 17 y las dependencias y levanta Cassandra 5.0 single-node (`infra/colab_setup.sh`).
3. Ejecuta `python -m src.pipeline run`: Landing → Bronze → Silver → Gold → Cassandra.
4. Muestra los resultados de las consultas CQL y corre las pruebas.
5. Descarga la evidencia generada (`evidence/e2/`).

In [ ]:
BRANCH = "e2"
REPO_URL = "https://github.com/smdevesa/tpe-big-data.git"
REPO_DIR = "/content/tpe-big-data"

In [ ]:
import os

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --format='commit %h · %ci · %s'

In [ ]:
!bash infra/colab_setup.sh
assert _exit_code == 0, "Falló la instalación"

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [ ]:
!python -m src.pipeline run
assert _exit_code == 0, "El pipeline falló: ver evidence/e2/run.log"

In [ ]:
!cat evidence/e2/cql_results.txt

In [ ]:
!bash -c "set -o pipefail; python -m pytest -q tests --junitxml=evidence/e2/pytest.xml 2>&1 | tee evidence/e2/pytest.txt"
assert _exit_code == 0, "Fallaron pruebas"

In [ ]:
!cd evidence && zip -qr /content/evidence_e2.zip e2
from google.colab import files
files.download("/content/evidence_e2.zip")